In [1]:
import os
os.chdir('/workspace/03610627-b381-4cb3-9393-d1a3a0d57d46')
print(os.listdir('.'))


['lchi5_cypari_worker.py', '.config', 'lmfdb_Lchi5.json', '-PROMPT-v6-DATASET.md', '.prompts', '.kernel_llm_logs_1.txt', 'memory']


In [2]:
import json
with open('lmfdb_Lchi5.json') as f:
 txt = f.read()
# Skip comments
data = json.loads(txt[txt.index('{'):])
print('num zeros:', len(data['positive_zeros']))
print('first 5:', data['positive_zeros'][:5])
print('last:', data['positive_zeros'][-1])


num zeros: 129
first 5: ['6.64845334472771471612327845997', '9.831444432886669616348321347458', '11.95884562608351453026565868826', '16.03382112838423567459325378224', '17.566994292325555202701595268144']
last: 199.09102170624125905531999800239


In [3]:
import cypari2
print(cypari2.__version__ if hasattr(cypari2, '__version__') else 'ok')
import multiprocessing as mp
print('cpus:', mp.cpu_count())


ModuleNotFoundError: No module named 'cypari2'

In [4]:
import subprocess
r = subprocess.run(['pip', 'install', 'cypari2', 'cysignals'], capture_output=True, text=True, timeout=300)
print(r.stdout[-2000:])
print('STDERR:', r.stderr[-1500:])



STDERR: 


In [5]:
import cypari2, multiprocessing as mp
print('cpus:', mp.cpu_count())
pari = cypari2.Pari()
pari.allocatemem(10**8)
pari.default("realprecision", 80)
pari("G = znstar(5, 1)")
pari("L = lfuncreate([G, [2]])")
import time
t0 = time.time()
z = pari.lfunzeros(pari("L"), [0, 30])
print('elapsed:', time.time()-t0)
print('zeros:', list(z)[:5])


cpus: 8
PARI stack size set to 100000000 bytes, maximum size set to 100003840
elapsed: 0.015464305877685547
zeros: [6.648453344727714717, 9.831444432886669616, 11.958845626083514530, 16.033821128384235675, 17.566994292325555203]


In [6]:
# Quick benchmark of how many zeros per unit t at various heights, and time
import time, cypari2
pari = cypari2.Pari()
pari.allocatemem(10**8)
pari.default("realprecision", 80)
pari("G = znstar(5, 1)")
pari("L = lfuncreate([G, [2]])")

# Estimate density: ~ N(T) ~ (T/2π) log(T) for L of conductor 5; roughly
# more precisely: count zeros in some sample windows
for t0, t1 in [(0,100),(500,600),(1000,1100),(2000,2100),(3000,3100),(4000,4100)]:
 s = time.time()
 z = pari.lfunzeros(pari("L"), [t0, t1])
 el = time.time()-s
 print(f't=[{t0},{t1}]: nz={len(z)} elapsed={el:.2f}s')


PARI stack size set to 100000000 bytes, maximum size set to 100003840
t=[0,100]: nz=54 elapsed=0.05s
t=[500,600]: nz=96 elapsed=1.02s
t=[1000,1100]: nz=107 elapsed=4.02s
t=[2000,2100]: nz=118 elapsed=24.26s
t=[3000,3100]: nz=124 elapsed=47.69s
t=[4000,4100]: nz=128 elapsed=49.24s


In [7]:
# Need to estimate Tmax so cumulative zero count is >= 5000.
# N(T) for Dirichlet L of conductor q=5 ~ (T/2π)(log(qT/2π) - 1)
# Use this to invert
import math
def N_approx(T, q=5):
 if T<=1: return 0
 return (T/(2*math.pi))*(math.log(q*T/(2*math.pi)) - 1) + 7/8

for T in [3000, 3500, 3800, 4000, 4200, 4400, 4500]:
 print(T, N_approx(T))


3000 3237.0974328781563
3500 3862.336287722044
3800 4243.05536322309
4000 4498.982339316029
4200 4756.501528649516
4400 5015.537052716108
4500 5145.601642247403


In [8]:
# Tmax ~ 4400 should cover 5000 zeros. We'll use 4500 for safety.
# Per-100 timing estimate (from above): roughly linear in t height.
# Total work estimate: integrate ~ ax+b. Roughly t=100->0.05, 1000->4, 2000->24, 3000->48, 4000->49.
# Looks like time per 100 t saturates around ~50s past t=3000.
# Total work T=0..4500: integral of about ... very rough: ~50s * 45 = 2250s. With 8 workers ideally ~280s.
# But timing scales further with PARI overhead per call; smaller intervals at higher t balance load.

# Plan: non-uniform partition. Allocate width w(t) ~ 1/cost(t). Cost(t) roughly ~ t at low t, capped near high t.
# Simpler: use power law width w(t) proportional to 1/sqrt(t+10) (so smaller intervals at higher t).
# Then dispatch to 8 workers via Pool.imap_unordered.

# Compute partition
import math
Tmax = 4500
# Width: choose so each interval has roughly equal cost. From timing, cost rate per unit t:
# t=50: 0.05/100=0.0005, t=550: 1.02/100=0.01, t=1050: 0.04, t=2050: 0.24, t=3050: 0.48, t=4050: 0.49
# Approximate cost rate c(t) = min(0.5, 0.0001*t^1.2) or similar.
# We'll use empirical: c(t) = max(0.0005, 0.00013 * t) capped at 0.5
def cost_rate(t):
 return min(0.5, max(0.0005, 0.00013*t))

# Build partition with cumulative cost segments of target equal cost.
# Total cost:
import numpy as np
ts = np.linspace(0, Tmax, 10000)
cs = np.array([cost_rate(t) for t in ts])
total = np.trapz(cs, ts)
print('estimated total cost (sec on 1 worker):', total)
# Choose ~64 segments => 8 per worker
nseg = 80
target = total/nseg
edges = [0.0]
acc = 0.0
last_i = 0
for i in range(1, len(ts)):
 acc += 0.5*(cs[i-1]+cs[i])*(ts[i]-ts[i-1])
 if acc >= target:
 edges.append(ts[i])
 acc = 0.0
edges.append(Tmax)
edges = sorted(set([round(e,2) for e in edges]))
print('nseg actual:', len(edges)-1)
print('first 10 edges:', edges[:10])
print('last 10 edges:', edges[-10:])
# widths
widths = [edges[i+1]-edges[i] for i in range(len(edges)-1)]
print('widths first/mid/last:', widths[0], widths[len(widths)//2], widths[-1])


AttributeError: module 'numpy' has no attribute 'trapz'

In [9]:
import math
import numpy as np

Tmax = 4500
def cost_rate(t):
 return min(0.5, max(0.0005, 0.00013*t))

ts = np.linspace(0, Tmax, 10000)
cs = np.array([cost_rate(t) for t in ts])
total = float(np.trapezoid(cs, ts))
print('estimated total cost (sec on 1 worker):', total)

nseg = 80
target = total/nseg
edges = [0.0]
acc = 0.0
for i in range(1, len(ts)):
 acc += 0.5*(cs[i-1]+cs[i])*(ts[i]-ts[i-1])
 if acc >= target:
 edges.append(float(ts[i]))
 acc = 0.0
if edges[-1] < Tmax:
 edges.append(Tmax)
edges = sorted(set([round(e,3) for e in edges]))
print('nseg actual:', len(edges)-1)
widths = [edges[i+1]-edges[i] for i in range(len(edges)-1)]
print('first 8 widths:', [round(w,2) for w in widths[:8]])
print('last 8 widths:', [round(w,2) for w in widths[-8:]])


estimated total cost (sec on 1 worker): 1288.4625015494348
nseg actual: 80
first 8 widths: [498.2, 206.12, 158.42, 133.66, 117.46, 106.21, 97.66, 90.91]
last 8 widths: [32.4, 32.4, 32.4, 32.4, 32.4, 32.4, 32.4, 20.25]


In [10]:
# Good partition. Now write the standalone script. Use multiprocessing with 'spawn' to avoid PARI fork issues.
# Save script and worker.
import os

script = r'''#!/usr/bin/env python3
"""Generate L(s, chi_4 mod 5) zeros at N=5000, dps=80 via non-uniform partition + multiprocessing."""
import os, sys, time, math, json
import numpy as np
import multiprocessing as mp

WORKDIR = "/workspace/03610627-b381-4cb3-9393-d1a3a0d57d46"
sys.path.insert(0, WORKDIR)

from lchi5_cypari_worker import worker as base_worker

DPS = 80
N_TARGET = 5000
TMAX = 4500.0
N_SEG = 80
N_WORKERS = 8

def cost_rate(t):
 return min(0.5, max(0.0005, 0.00013*t))

def build_partition(tmax=TMAX, nseg=N_SEG):
 ts = np.linspace(0, tmax, 10000)
 cs = np.array([cost_rate(t) for t in ts])
 total = float(np.trapezoid(cs, ts))
 target = total / nseg
 edges = [0.0]
 acc = 0.0
 for i in range(1, len(ts)):
 acc += 0.5*(cs[i-1]+cs[i])*(ts[i]-ts[i-1])
 if acc >= target:
 edges.append(float(ts[i]))
 acc = 0.0
 if edges[-1] < tmax:
 edges.append(tmax)
 edges = sorted(set([round(e,3) for e in edges]))
 return edges

def main():
 edges = build_partition()
 intervals = [(edges[i], edges[i+1], DPS, i) for i in range(len(edges)-1)]
 print(f"[{time.strftime('%H:%M:%S')}] Built {len(intervals)} intervals over [0,{TMAX}]", flush=True)
 print(f" first widths: {[round(intervals[i][1]-intervals[i][0],2) for i in range(3)]}", flush=True)
 print(f" last widths : {[round(intervals[-1-i][1]-intervals[-1-i][0],2) for i in range(3)]}", flush=True)

 t_start = time.time()
 ctx = mp.get_context("spawn")
 results = {}
 with ctx.Pool(N_WORKERS) as pool:
 for ridx, tmin, tmax, elapsed, zs in pool.imap_unordered(base_worker, intervals):
 results[ridx] = (tmin, tmax, elapsed, zs)
 done = len(results)
 total_z = sum(len(v[3]) for v in results.values())
 print(f"[{time.strftime('%H:%M:%S')}] seg {ridx:3d} [{tmin:7.2f},{tmax:7.2f}] nz={len(zs):4d} t={elapsed:6.1f}s | done {done}/{len(intervals)} totalZ={total_z}", flush=True)

 total_time = time.time() - t_start
 print(f"All workers finished in {total_time:.1f}s", flush=True)

 # Merge in order
 all_zeros = []
 for idx in sorted(results.keys()):
 tmin, tmax, el, zs = results[idx]
 all_zeros.extend(zs)
 print(f"Total raw zeros collected: {len(all_zeros)}", flush=True)

 # Sort by float value, de-duplicate (within 1e-9), truncate to N_TARGET
 pairs = sorted(((float(s.replace(' ', '')), s) for s in all_zeros), key=lambda x: x[0])
 dedup = []
 for fv, s in pairs:
 if dedup and abs(fv - dedup[-1][0]) < 1e-9:
 continue
 dedup.append((fv, s))
 print(f"After dedup: {len(dedup)} zeros", flush=True)

 if len(dedup) < N_TARGET:
 print(f"ERROR: only {len(dedup)} zeros, need {N_TARGET}", flush=True)
 sys.exit(2)

 final = [s for fv, s in dedup[:N_TARGET]]
 arr = np.array(final, dtype='<U81')
 out_path = os.path.join(WORKDIR, "lchi5_zeros_N5000_dps80.npy")
 np.save(out_path, arr)
 print(f"Saved {out_path} shape={arr.shape} dtype={arr.dtype}", flush=True)

 # Validation vs LMFDB
 with open(os.path.join(WORKDIR, "lmfdb_Lchi5.json")) as f:
 txt = f.read()
 lmfdb = json.loads(txt[txt.index('{'):])["positive_zeros"]
 print(f"Validating against {len(lmfdb)} LMFDB zeros...", flush=True)
 max_err = 0.0
 for i, ref in enumerate(lmfdb):
 ours = final[i]
 # compare common prefix precision -- LMFDB has ~28 digits, ours ~65
 diff = abs(float(ours) - float(ref))
 if diff > max_err: max_err = diff
 if i < 5 or diff > 1e-15:
 print(f" [{i}] ref={ref[:30]} ours={ours[:30]} |diff|={diff:.2e}", flush=True)
 print(f"Max |diff| over first {len(lmfdb)} LMFDB zeros: {max_err:.3e}", flush=True)

if __name__ == "__main__":
 main()
'''

with open('generate_lchi5_dps80.py', 'w') as f:
 f.write(script)
print('written:', os.path.getsize('generate_lchi5_dps80.py'))


written: 3813


In [11]:
# First do a quick smoke test of the worker via spawn pool, single small interval
import multiprocessing as mp, sys, time
sys.path.insert(0, '/workspace/03610627-b381-4cb3-9393-d1a3a0d57d46')
from lchi5_cypari_worker import worker

ctx = mp.get_context('spawn')
with ctx.Pool(2) as pool:
 out = list(pool.imap_unordered(worker, [(0.0, 20.0, 80, 0), (20.0, 40.0, 80, 1)]))
for o in out:
 print(o[:4], o[4][:2], '...')


TimeoutError: Code execution timed out after 900.0 seconds

In [12]:
# Cell hung - likely the spawned children inherited a stuck state, OR they need to import lchi5_cypari_worker 
# from spawn context but PARI is initialising slowly with realprec=80?
# Run as standalone subprocess instead so we don't pollute the kernel.
import subprocess, sys, os
test_script = r'''
import sys, time
sys.path.insert(0, '/workspace/03610627-b381-4cb3-9393-d1a3a0d57d46')
import multiprocessing as mp
from lchi5_cypari_worker import worker
if __name__ == "__main__":
 ctx = mp.get_context('spawn')
 t0 = time.time()
 with ctx.Pool(2) as pool:
 out = list(pool.imap_unordered(worker, [(0.0, 20.0, 80, 0), (20.0, 40.0, 80, 1)]))
 print('done in', time.time()-t0, 's')
 for o in out:
 print(o[:4])
 print(' first zero:', o[4][:2])
'''
with open('/tmp/test_worker.py','w') as f: f.write(test_script)
r = subprocess.run([sys.executable, '/tmp/test_worker.py'], capture_output=True, text=True, timeout=300)
print('STDOUT:', r.stdout)
print('STDERR:', r.stderr[-2000:])
print('RC:', r.returncode)


TimeoutError: Code execution timed out after 900.0 seconds

In [13]:
# Spawn pool seems to hang. Let's check.
# Maybe importing cypari2 in spawned children is slow but shouldn't hang.
# Let's try a single subprocess with fork-via-exec pattern. Try without Pool, using Process.
import subprocess, sys
ts = r'''
import sys, time
sys.path.insert(0, '/workspace/03610627-b381-4cb3-9393-d1a3a0d57d46')
from lchi5_cypari_worker import worker
if __name__ == "__main__":
 t0=time.time()
 r = worker((0.0, 20.0, 80, 0))
 print('elapsed:', time.time()-t0)
 print(r[:4])
 print(r[4][:3])
'''
with open('/tmp/tw_single.py','w') as f: f.write(ts)
r = subprocess.run([sys.executable, '/tmp/tw_single.py'], capture_output=True, text=True, timeout=180)
print('STDOUT:', r.stdout)
print('STDERR:', r.stderr[-1500:])
print('RC:', r.returncode)


TimeoutError: Code execution timed out after 900.0 seconds

In [14]:
# Notebook is stuck on stale subprocesses. Try resetting kernel.
print("alive")


TimeoutError: Code execution timed out after 25.0 seconds